# **Travaux Pratiques : RAG Multimodal & Agentique**
### **Conception, Indexation et Interrogation Autonome (Textes & Images)**
**Document d'étude :** *Attention Is All You Need* (Vaswani et al., 2017)

---

### **Objectifs du TP :**
1. **Étape 1 :** Extraction du contenu multimodal (texte intégral et images/figures intégrées du PDF).
2. **Étape 2 :** Vectorisation dans un espace latent commun avec le modèle vision-langage **CLIP** (`ViT-B-32`) et normalisation $\ell_2$.
3. **Étape 3 :** Indexation et stockage persistant dans la base vectorielle **ChromaDB** avec identifiant de document SHA-256 et métadonnées riches.
4. **Étape 4 :** Recherche multimodale par similarité cosinus (requêtes texte-texte, texte-image, image-image, image-texte).
5. **Étape 5 :** Transformation du moteur de recherche en outil agentique (`retriever_tool`) typé et documenté pour un LLM.
6. **Étape 6 :** Construction d'un Agent IA autonome basé sur le patron **ReAct** (*Reasoning $\rightarrow$ Action $\rightarrow$ Observation*) compilé avec **LangGraph** (Architecture MIND & BODY).
7. **Étape 7 :** Tests, validation empirique sur requêtes hybrides et analyse dynamique du comportement de l'agent.

---
### **Architecture ReAct & LangGraph (MIND ⇄ BODY) :**

```
   ┌────────────────────────────────────────────────────────┐
   │             Utilisateur / Requête Multimodale          │
   └───────────────────────────┬────────────────────────────┘
                               │
                               ▼
   ┌────────────────────────────────────────────────────────┐
   │             MIND (Cerveau - Raisonnement LLM)           │
   │      - Analyse la question                             │
   │      - Formule une Thought (Pensée)                    │
   │      - Décide d'une Action (tool call: retriever_tool) │
   └───────────┬────────────────────────────────▲───────────┘
               │                                │
     (Action : appel d'outil)            (Observation)
               │                                │
               ▼                                │
   ┌────────────────────────────────────────────┴───────────┐
   │             BODY (Corps - ToolNode)                    │
   │  Exécute retriever_tool sur ChromaDB (CLIP ViT-B-32)    │
   └────────────────────────────────────────────────────────┘
               │ (Aucun appel d'outil / Réponse prête)
               ▼
   ┌────────────────────────────────────────────────────────┐
   │             END (Synthèse Finale Documentée)           │
   │  Réponse structurée avec citations [PDF, p. X]         │
   │  et affichage des figures associées                     │
   └────────────────────────────────────────────────────────┘
```


## **0. Environnement et Dépendances**

Nous importons les bibliothèques requises pour l'ensemble du pipeline :
- `pymupdf` (`fitz`) : extraction du texte et des images brutes du PDF.
- `torch` & `open_clip` : modèle vision-langage CLIP pour l'espace latent partagé.
- `chromadb` : base de données vectorielle avec indexation HNSW et métrique cosinus.
- `PIL` (`Pillow`) & `IPython.display` : manipulation et rendu visuel des images.
- `langchain_core` & `langgraph` : définition des outils et compilation du graphe ReAct.


In [ ]:
from pathlib import Path
import os
import re
import json
import base64
import hashlib
from io import BytesIO
from typing import List, Dict, Any, Optional, Tuple, Literal

import pymupdf
import torch
import open_clip
from open_clip.tokenizer import SimpleTokenizer
import chromadb
from PIL import Image
from IPython.display import display, Markdown

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# Configuration matérielle
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ PyTorch Device configuré : {DEVICE}")

# Configuration des chemins
PDF_PATH = Path("attention is all you need.pdf")
if not PDF_PATH.is_file():
    PDF_PATH = Path("pdfs/attention is all you need.pdf")

OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"

TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Fichier PDF source : {PDF_PATH.resolve()}")
print(f"✓ Dossier de sortie   : {OUTPUT_DIR.resolve()}")


## **Étape 1 : Extraction du contenu multimodal (Textes & Images)**

Dans cette étape :
1. Nous ouvrons le PDF avec `pymupdf`.
2. Pour chaque page :
   - Nous extrayons le texte intégral ordonné (`page.get_text("text", sort=True)`) et le sauvegardons dans un fichier texte dédié (`page_XXX.txt`).
   - Nous identifions les images intégrées par leur référence XREF (`page.get_images(full=True)`), évitons les doublons au sein d'une même page, et sauvegardons les flux binaires (`page_XXX_image_YYY.png`).
3. Nous concaténons le texte complet dans `texte_complet.txt`.


In [ ]:
textes = []
nombre_images = 0
pages_sans_texte = []

with pymupdf.open(PDF_PATH) as document:
    nombre_pages = len(document)

    for numero_page, page in enumerate(document, start=1):
        # 1. Extraction du texte de la page
        texte = page.get_text("text", sort=True)
        fichier_texte = TEXT_DIR / f"page_{numero_page:03d}.txt"
        fichier_texte.write_text(texte, encoding="utf-8")
        textes.append(f"===== PAGE {numero_page} =====\n\n{texte}")

        if not texte.strip():
            pages_sans_texte.append(numero_page)

        # 2. Extraction des images de la page
        images_vues = set()
        for image_info in page.get_images(full=True):
            xref = image_info[0]
            if xref in images_vues:
                continue
            images_vues.add(xref)

            try:
                image = document.extract_image(xref)
                if not image:
                    continue

                extension = image["ext"]
                fichier_image = IMAGE_DIR / f"page_{numero_page:03d}_image_{xref}.{extension}"
                fichier_image.write_bytes(image["image"])
                nombre_images += 1
            except Exception as erreur:
                print(f"Erreur extraction image (page {numero_page}, xref {xref}) : {erreur}")

# Sauvegarde du document complet
(OUTPUT_DIR / "texte_complet.txt").write_text("\n\n".join(textes), encoding="utf-8")

print(f"✓ Extraction terminée avec succès !")
print(f"  - Pages traitées      : {nombre_pages}")
print(f"  - Images enregistrées : {nombre_images}")


### **Visualisation et vérification des données extraites**
Vérifions le texte extrait et affichons les figures du document (notamment Figure 1 : architecture globale, et Figure 2 : attention scaled dot-product et multi-têtes).


In [ ]:
print("--- APERÇU DU TEXTE EXTRAIT (Page 1) ---")
texte_complet = (OUTPUT_DIR / "texte_complet.txt").read_text(encoding="utf-8")
print(texte_complet[:1200] + "\n...\n")

print("--- FIGURES EXTRAITES DU PDF ---")
for fichier_img in sorted(IMAGE_DIR.iterdir()):
    print(f"Image : {fichier_img.name} ({fichier_img.stat().st_size / 1024:.1f} KB)")
    with Image.open(fichier_img) as img:
        apercu = img.copy()
        apercu.thumbnail((500, 400))
        display(apercu)

# Sauvegarde de référence pour la Figure 1 (Architecture Transformer)
fig1_cand = list(IMAGE_DIR.glob("page_003*.png"))
if fig1_cand:
    Image.open(fig1_cand[0]).save("architecture_transformer.png")
    print("✓ Image 'architecture_transformer.png' prête pour les requêtes de test.")


## **Étape 2 : Vectorisation dans un espace latent commun (Embeddings CLIP)**

### **Principe théorique :**
Le modèle **CLIP** (*Contrastive Language-Image Pretraining*) projette des modalités hétérogènes (textes et images) dans un **espace vectoriel latent partagé** $\mathbb{R}^d$ ($d=512$ pour ViT-B-32).

Pour assurer que le produit scalaire représente fidèlement la similarité cosinus, les vecteurs doivent obligatoirement être **normalisés selon la norme $\ell_2$** :
$$\mathbf{v}_{\text{norm}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2} = \frac{\mathbf{v}}{\sqrt{\sum_{i=1}^d v_i^2}}$$

### **Découpage textuel :**
Comme le tokeniseur CLIP limite les séquences à 77 tokens, nous découpons le texte en fragments de **60 tokens BPE** avec un **chevauchement de 10 tokens**.


In [ ]:
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

print(f"Chargement du modèle CLIP {MODEL_NAME} ({PRETRAINED})...")
model_clip, _, preprocess_clip = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED
)
model_clip = model_clip.to(DEVICE).eval()
tokenizer_clip = open_clip.get_tokenizer(MODEL_NAME)
bpe_tokenizer = SimpleTokenizer()

print("✓ Modèle CLIP chargé et prêt pour l'inférence.")

def decouper_texte(texte, taille=60, chevauchement=10):
    """Découper le texte en fragments de 60 tokens BPE avec chevauchement."""
    tokens = bpe_tokenizer.encode(texte)
    for debut in range(0, len(tokens), taille - chevauchement):
        fragment = bpe_tokenizer.decode(tokens[debut:debut + taille]).strip()
        if fragment:
            yield fragment
        if debut + taille >= len(tokens):
            break

@torch.inference_mode()
def encoder_textes(liste_textes):
    """Calculer et normaliser l'embedding CLIP pour un lot de textes."""
    tokens = tokenizer_clip(liste_textes).to(DEVICE)
    vecteurs = model_clip.encode_text(tokens).float()
    # Normalisation L2
    vecteurs = vecteurs / vecteurs.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vecteurs.cpu().tolist()

@torch.inference_mode()
def encoder_image(chemin_image):
    """Calculer et normaliser l'embedding CLIP pour une image."""
    with Image.open(chemin_image) as img:
        tenseur = preprocess_clip(img.convert("RGB"))
    tenseur = tenseur.unsqueeze(0).to(DEVICE)
    vecteur = model_clip.encode_image(tenseur).float()
    # Normalisation L2
    vecteur = vecteur / vecteur.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vecteur.cpu().tolist()[0]


## **Étape 3 : Indexation et stockage dans la base vectorielle ChromaDB**

1. **Initialisation de ChromaDB** : Création d'un client persistant sur disque (`chroma_pdf/`) et d'une collection avec métrique cosinus (`{"hnsw:space": "cosine"}`).
2. **Identifiant de document SHA-256** : Calcul d'un hash cryptographique pour garantir la cohérence des réindexations :
   $$\text{DocID} = \text{SHA-256}(\text{Fichier PDF})_{[:24]}$$
3. **Indexation avec métadonnées** :
   - Textes : `type="texte"`, `document_id`, `pdf`, `page`, `fragment`, `source`.
   - Images : `type="image"`, `document_id`, `pdf`, `page`, `source` (chemin local du fichier image).


In [ ]:
client_chroma = chromadb.PersistentClient(path="chroma_pdf")

collection = client_chroma.get_or_create_collection(
    name="pdf_clip_vit_b32",
    embedding_function=None,
    metadata={"hnsw:space": "cosine"}
)

# Calcul du document_id SHA-256
document_id = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:24]
print(f"✓ Identifiant SHA-256 du document : {document_id}")

# Réinitialisation ciblée pour idempotence
collection.delete(where={"document_id": document_id})

def extraire_numero_page(chemin):
    match = re.search(r"page_(\d+)", Path(chemin).stem)
    return int(match.group(1)) if match else -1

# 1. Indexation des fragments textuels
ids_textes, docs_textes, metas_textes = [], [], []
for fichier in sorted(TEXT_DIR.glob("page_*.txt")):
    contenu_page = fichier.read_text(encoding="utf-8")
    page_num = extraire_numero_page(fichier)

    for idx, fragment in enumerate(decouper_texte(contenu_page)):
        ids_textes.append(f"{document_id}:texte:{page_num}:{idx}")
        docs_textes.append(fragment)
        metas_textes.append({
            "type": "texte",
            "document_id": document_id,
            "pdf": PDF_PATH.name,
            "page": page_num,
            "fragment": idx,
            "source": str(fichier.resolve())
        })

# Envoi par lots de 64
LOT_TAILLE = 64
for deb in range(0, len(docs_textes), LOT_TAILLE):
    fin = deb + LOT_TAILLE
    collection.upsert(
        ids=ids_textes[deb:fin],
        documents=docs_textes[deb:fin],
        metadatas=metas_textes[deb:fin],
        embeddings=encoder_textes(docs_textes[deb:fin])
    )
print(f"✓ {len(docs_textes)} fragments de texte indexés.")

# 2. Indexation des images
extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff", ".jp2"}
nb_imgs_indexees = 0
for fichier_img in sorted(IMAGE_DIR.iterdir()):
    if not fichier_img.is_file() or fichier_img.suffix.lower() not in extensions:
        continue

    vecteur_img = encoder_image(fichier_img)
    page_num = extraire_numero_page(fichier_img)

    collection.upsert(
        ids=[f"{document_id}:image:{fichier_img.name}"],
        embeddings=[vecteur_img],
        documents=[f"Image extraite du PDF : {fichier_img.name}"],
        metadatas=[{
            "type": "image",
            "document_id": document_id,
            "pdf": PDF_PATH.name,
            "page": page_num,
            "source": str(fichier_img.resolve())
        }]
    )
    nb_imgs_indexees += 1

print(f"✓ {nb_imgs_indexees} images indexées.")
print(f"✓ Total dans la collection ChromaDB : {collection.count()} éléments.")


## **Étape 4 : Implémentation de la recherche multimodale (Retrieval)**

Nous implémentons une fonction de recherche par **similarité cosinus** :
$$\text{Similarité}(u, v) = 1 - \text{Distance Cosinus}(u, v)$$

La fonction accepte :
- Soit une requête textuelle (`texte="..."`).
- Soit une image de requête (`image="..."`).
- Un filtre optionnel `type_resultat` (`"texte"`, `"image"`, ou `None`).
- Le nombre $k$ de résultats les plus proches.


In [ ]:
def rechercher(texte=None, image=None, type_resultat=None, k=5):
    """Recherche multimodale par similarité cosinus dans ChromaDB."""
    if (texte is None) == (image is None):
        raise ValueError("Fournissez soit un texte, soit une image.")
    if type_resultat not in (None, "texte", "image"):
        raise ValueError("type_resultat doit être 'texte', 'image' ou None.")

    total = collection.count()
    if total == 0:
        raise ValueError("La collection est vide.")

    # Vectorisation selon la modalité de la requête
    if texte is not None:
        vecteur = encoder_textes([texte])[0]
    else:
        vecteur = encoder_image(image)

    options = {}
    if type_resultat is not None:
        options["where"] = {"type": type_resultat}

    return collection.query(
        query_embeddings=[vecteur],
        n_results=min(k, total),
        include=["documents", "metadatas", "distances"],
        **options
    )

def afficher_resultats(resultats):
    """Affiche les résultats textuels et le rendu des images trouvées."""
    for rang, (doc, meta, dist) in enumerate(
        zip(resultats["documents"][0], resultats["metadatas"][0], resultats["distances"][0]),
        start=1
    ):
        similarite = 1.0 - dist
        print(f"\n[{rang}] Type: {meta['type'].upper()} | Page: {meta['page']} | Similarité cosinus: {similarite:.3f}")
        if meta["type"] == "image":
            chemin_img = Path(meta["source"])
            if chemin_img.is_file():
                with Image.open(chemin_img) as im:
                    apercu = im.convert("RGB")
                    apercu.thumbnail((450, 350))
                    display(apercu)
        else:
            print(f"Extrait : {doc[:220]}...")


### **Validation des 4 modes de recherche multimodale :**


In [ ]:
print("=== TEST A : Texte -> Textes (Multi-Head Attention) ===")
res_a = rechercher(texte="multi-head attention mechanism and projections", type_resultat="texte", k=3)
afficher_resultats(res_a)

print("\n=== TEST B : Texte -> Images (Transformer Diagram) ===")
res_b = rechercher(texte="transformer model architecture diagram encoder decoder", type_resultat="image", k=2)
afficher_resultats(res_b)

print("\n=== TEST C : Image -> Images (Recherche par schéma) ===")
res_c = rechercher(image="architecture_transformer.png", type_resultat="image", k=2)
afficher_resultats(res_c)

print("\n=== TEST D : Image -> Textes (Explication à partir du schéma) ===")
res_d = rechercher(image="architecture_transformer.png", type_resultat="texte", k=3)
afficher_resultats(res_d)


## **Étape 5 : Transformation du Retriever en Outil Agentique (Tool)**

Pour permettre à un agent IA d'interroger la base de données vectorielle de manière autonome, nous encapsulons la fonction de recherche dans un **Outil LangChain** (`@tool`).

### **Composants de l'Outil :**
- **Nom de l'outil** : `retriever_tool`
- **Typage strict** : `Literal["tous", "texte", "image"]` pour le ciblage modal.
- **Description sémantique** : Explique au LLM quand et comment utiliser l'outil pour chercher des explications, formules ou figures.
- **Format de sortie JSON** : Renvoie les identifiants, extraits textuels, numéros de page et chemins d'images avec leurs similarités.


In [ ]:
@tool
def retriever_tool(
    question: str,
    type_resultat: Literal["tous", "texte", "image"] = "tous",
    k: int = 6
) -> str:
    """
    Consulter la base vectorielle multimodale du document PDF 'Attention Is All You Need'.
    Permet d'extraire des explications textuelles, équations, résultats empiriques et figures/diagrammes d'architecture.

    Paramètres :
    - question : Requête en texte clair décrivant le concept, mécanisme ou figure recherché.
    - type_resultat : "tous" (texte et images), "texte" (passages textuels seuls), "image" (figures/diagrammes seuls).
    - k : Nombre de résultats les plus proches à retourner (1 à 10).
    """
    filtre = None if type_resultat == "tous" else type_resultat
    k_ajuste = max(1, min(k, 10))

    try:
        resultats = rechercher(texte=question, type_resultat=filtre, k=k_ajuste)
    except Exception as e:
        return json.dumps({"erreur": str(e)}, ensure_ascii=False)

    sources = []
    for idx, (doc_id, doc, meta, dist) in enumerate(
        zip(resultats["ids"][0], resultats["documents"][0], resultats["metadatas"][0], resultats["distances"][0]),
        start=1
    ):
        similarity = round(1.0 - dist, 3)
        sources.append({
            "reference": f"S{idx}",
            "id": doc_id,
            "type": meta["type"],
            "pdf": meta["pdf"],
            "page": meta["page"],
            "similarite": similarity,
            "source_path": meta["source"],
            "extrait": doc if meta["type"] == "texte" else f"Image file: {meta['source']}"
        })

    return json.dumps({
        "question": question,
        "nombre_sources": len(sources),
        "sources": sources
    }, ensure_ascii=False)

print(f"✓ Outil '{retriever_tool.name}' configuré avec succès.")
print(f"  Description : {retriever_tool.description[:100]}...")


## **Étape 6 : Construction de l'Agent IA (Pattern ReAct & LangGraph)**

L'architecture de l'agent est modélisée selon le patron **ReAct** (*Reasoning $\rightarrow$ Action $\rightarrow$ Observation*) articulant :
1. **Le Cerveau (MIND)** : Nœud de raisonnement exécuté par le LLM. Il évalue la requête, élabore une pensée, et génère un appel d'outil structuré (`retriever_tool`).
2. **Le Corps (BODY)** : Nœud d'exécution `ToolNode([retriever_tool])` qui effectue la recherche dans la base vectorielle et produit l'observation.
3. **Le Graphe de flux (LangGraph)** :
   - `START ──► mind`
   - `mind ──(tools_condition)──► body` si un appel d'outil est requis.
   - `body ──► mind` pour observer le résultat et formuler la synthèse.
   - `mind ──(tools_condition)──► END` quand la réponse finale est prête.


In [ ]:
SYSTEM_PROMPT_AGENT = """Tu es un assistant scientifique expert en Deep Learning et architectures Transformer.
Tu disposes de l'outil `retriever_tool` pour interroger l'article 'Attention Is All You Need'.

Règles de fonctionnement :
1. Pour toute question sur le papier, appelle `retriever_tool` avec des mots-clés précis en anglais (plus efficace avec CLIP).
2. Choisis type_resultat="texte" pour des explications théoriques, "image" pour des figures/diagrammes, ou "tous" pour une recherche mixte.
3. Appuie tes affirmations exclusivement sur les sources retournées par l'outil.
4. Dans ta réponse finale, cite systématiquement les pages du document : [Attention is all you need.pdf, p. X].
5. N'invente aucun chiffre, hyperparamètre ou référence absente des sources."""

def construire_agent_langgraph(model=None):
    """Compile le graphe ReAct avec LangGraph (MIND & BODY)."""
    tools = [retriever_tool]
    body_node = ToolNode(tools)

    if model is None:
        api_key = os.environ.get("OPENAI_API_KEY")
        if api_key and api_key != "....":
            from langchain_openai import ChatOpenAI
            model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        else:
            # Moteur ReAct local de fallback autonome pour tester sans clé externe
            class LocalReActLLM:
                def invoke(self, messages):
                    last_msg = messages[-1]
                    if isinstance(last_msg, HumanMessage) or (isinstance(last_msg, dict) and last_msg.get("role") == "user"):
                        q_text = last_msg.content if isinstance(last_msg, HumanMessage) else last_msg.get("content", "")
                        type_res = "image" if any(w in q_text.lower() for w in ["figure", "image", "diagramme", "schéma"]) else "tous"
                        return AIMessage(
                            content="Je consulte la base vectorielle pour identifier les passages et diagrammes pertinents.",
                            tool_calls=[{
                                "name": "retriever_tool",
                                "args": {"question": q_text, "type_resultat": type_res, "k": 4},
                                "id": "call_react_1"
                            }]
                        )
                    elif isinstance(last_msg, ToolMessage):
                        try:
                            data = json.loads(last_msg.content)
                            sources = data.get("sources", [])
                            res_text = "D'après l'article *Attention Is All You Need* :\n\n"
                            for s in sources:
                                if s["type"] == "texte":
                                    res_text += f"- **Extrait (page {s['page']})** : {s['extrait'][:180]}... [Attention is all you need.pdf, p. {s['page']}]\n"
                                else:
                                    res_text += f"- **Figure (page {s['page']})** : {Path(s['source_path']).name} [Attention is all you need.pdf, p. {s['page']}]\n"
                            return AIMessage(content=res_text)
                        except Exception:
                            return AIMessage(content="Données documentaires récupérées avec succès.")
                    return AIMessage(content="Terminé.")
            model = LocalReActLLM()

    # Construction du StateGraph
    workflow = StateGraph(MessagesState)

    def mind_node(state: MessagesState):
        messages = state["messages"]
        if not any(isinstance(m, SystemMessage) for m in messages):
            messages = [SystemMessage(content=SYSTEM_PROMPT_AGENT)] + messages
        reponse = model.invoke(messages)
        return {"messages": [reponse]}

    workflow.add_node("mind", mind_node)
    workflow.add_node("body", body_node)

    workflow.add_edge(START, "mind")
    workflow.add_conditional_edges("mind", tools_condition, {"tools": "body", END: END})
    workflow.add_edge("body", "mind")

    return workflow.compile()

agent_react = construire_agent_langgraph()
print("✓ Agent LangGraph ReAct compilé avec succès !")


## **Étape 7 : Tests, validation et analyse du comportement de l'Agent RAG**

Nous soumettons des requêtes complexes nécessitant :
1. Une recherche hybride (textes conceptuels et figures).
2. Une analyse des décisions d'appel d'outil prises par le MIND.
3. L'affichage coordonné de la réponse textuelle et des images associées.


In [ ]:
def interroger_agent(agent, question, image_requete=None):
    """Exécute le cycle ReAct et affiche la trace d'exécution et les résultats."""
    contenu_requete = question
    if image_requete:
        contenu_requete += f" [Requête enrichie par l'image : {Path(image_requete).name}]"

    state_init = {"messages": [HumanMessage(content=contenu_requete)]}
    execution = agent.invoke(state_init)

    messages = execution["messages"]
    reponse_finale = messages[-1].content
    appels_outils = []
    images_retrouvees = []

    for msg in messages:
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            appels_outils.extend(msg.tool_calls)
        elif isinstance(msg, ToolMessage):
            try:
                data = json.loads(msg.content)
                for src in data.get("sources", []):
                    if src.get("type") == "image":
                        p_img = Path(src.get("source_path"))
                        if p_img.is_file() and p_img not in [im["path"] for im in images_retrouvees]:
                            images_retrouvees.append({
                                "path": p_img,
                                "page": src.get("page"),
                                "similarite": src.get("similarite")
                            })
            except Exception:
                pass

    display(Markdown(f"### ❓ Question : {question}"))
    if image_requete:
        print(f"Image fournie : {image_requete}")

    print("\n--- ⚙️ TRACE D'EXÉCUTION REACT (MIND ⇄ BODY) ---")
    for appel in appels_outils:
        print(f"Action d'outil : {appel['name']} | Paramètres : {json.dumps(appel['args'], ensure_ascii=False)}")

    display(Markdown(f"### 📝 Réponse Finale de l'Agent\n\n{reponse_finale}"))

    if images_retrouvees:
        display(Markdown(f"### 🖼️ Figures Retrouvées ({len(images_retrouvees)})"))
        for im in images_retrouvees:
            print(f"- Figure Page {im['page']} (Similarité cosinus : {im['similarite']}) : {im['path'].name}")
            with Image.open(im["path"]) as f_img:
                ap = f_img.copy()
                ap.thumbnail((500, 380))
                display(ap)

    return execution


In [ ]:
# Test 1 : Requête hybride (Théorie & Schéma d'attention)
sortie1 = interroger_agent(
    agent_react,
    question="Explique le mécanisme d'attention Multi-Head Attention et retrouve la figure correspondante."
)


In [ ]:
# Test 2 : Requête basée sur une image en entrée (Figure 1 de l'architecture)
sortie2 = interroger_agent(
    agent_react,
    question="Identifie cette figure d'architecture et décris le rôle de l'encodeur et du décodeur.",
    image_requete=Path("architecture_transformer.png")
)


## **Conclusion et Déploiement Streamlit**

Ce TP a permis d'implémenter de bout en bout :
1. **L'extraction multimodale** du document PDF avec `pymupdf`.
2. **L'encodage multimodal normalisé** avec le modèle CLIP `ViT-B-32`.
3. **L'indexation vectorielle persistante** et reproductible avec **ChromaDB** et hachage SHA-256.
4. **La recherche croisée par similarité cosinus** (texte/image).
5. **L'encapsulation sous forme d'outil agentique** `retriever_tool`.
6. **L'orchestration d'un Agent ReAct** articulant MIND et BODY avec **LangGraph**.
7. **La validation empirique** sur des requêtes documentaires et multimodales complexes.

> Pour lancer l'interface web interactive :
> ```bash
> streamlit run app_streamlit.py
> ```
